# **Lazizbek Sadullaev**

# **Final Project, Math 548**

#**Example problems to compare the methods:**


**Solve the following systems of equations first using the Jacobi iteration method, then Gauss-Seidel iteration method, and finally new introduced Multi-scale method. In each case, compare the numerical results.**


1.   [ 1   1      0][x1]   [     2]

     [ 0  100     1][x2] = [   101]

     [ 1   0  10000][x3]   [ 10001]


2.   [   1   (1/2)  (1/3)][x1]   [ 1]
    
     [(1/2)  (1/3)  (1/4)][x2] = [ 1]
    
     [(1/3)  (1/4)  (1/5)][x3]   [ 1]

# **Solving Ax=b matrix equation:**
[Source:](https://stackoverflow.com/questions/22163113/matrix-multiplication-solve-ax-b-solve-for-x)

In [ ]:
# To solve Ax=b, We start by constructing the arrays for A and b.

import numpy as np
A = np.array([[ 1,  1,  0],
              [-2,  0,  2],
              [ 4, -4,  0]])
b = np.transpose(np.array([ (-7/16),  (11/8),  0]))
# To solve the system we do

x = np.linalg.solve(A,b)
print("Real solution: ")
print(x)

# **Finding eigenstuff of a matrix:**
[Source:](https://pythonnumericalmethods.berkeley.edu/notebooks/chapter15.04-Eigenvalues-and-Eigenvectors-in-Python.html)

In [ ]:
# To find the eigenvalues and eigenvectors we do
import numpy as np
from numpy.linalg import eig
a = np.array([[0, 2],
              [2, 3]])
w,v=eig(a)
print('E-value:', w)
print('E-vector', v)

E-value: [-1.  4.]
E-vector [[-0.89442719 -0.4472136 ]
 [ 0.4472136  -0.89442719]]


# **Example 1. Finding exact solution:**

In [ ]:
# To solve Ax=b, We start by constructing the arrays for A and b.

import numpy as np
A = np.array([[ 1,  1,  0],
              [ 0,  100,  1],
              [ 1,  0,  10000]])
b = np.transpose(np.array([ 2, 101, 10001]))
# To solve the system we do

x = np.linalg.solve(A,b)
print("Real solution: ")
print(x)

Real solution: 
[1. 1. 1.]


# **Solving Example 1 using Jacobi, Gauss-Siedel, and Multi-scale methods for comparison:**

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import LinearOperator, gmres

# Define the coefficient matrix A and the right-hand side vector b
A = np.array([
    [1, 1, 0],
    [0, 100, 1],
    [1, 0, 10000]
])

b = np.array([2, 101, 10001])

# Define the exact solution for comparison
x_exact = np.linalg.solve(A, b)

# Define a function for the Jacobi method
def jacobi(A, b, x0, tol=1e-10, max_iterations=1000):
    n = len(b)
    x = x0.copy()
    D = np.diag(A)
    R = A - np.diagflat(D)
    for i in range(max_iterations):
        x_new = (b - np.dot(R, x)) / D
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            break
        x = x_new
    return x, i+1

# Define a function for the Gauss-Seidel method
def gauss_seidel(A, b, x0=None, tol=1e-10, max_iterations=1000):
    n = len(b)
    if x0 is None:
        x0 = np.zeros(n)

    x = x0.copy()
    D = np.diag(np.diag(A))
    L = np.tril(A, -1)
    U = np.triu(A, 1)

    for k in range(max_iterations):
        x_new = np.dot(np.linalg.inv(D + L), b - np.dot(U, x))

        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            return x_new, k
        x = x_new

    return x, max_iterations


# Define a function for the Multi-scale method using the Gauss-Seidel, Jacobi, GMRES methods
def multi_scale(A, b, tol=1e-10, max_iterations=1000):
    D = np.linalg.norm(A, axis=0)  # Compute the Diagonal Scaling Matrix D
    D = np.diag(D)
    A_tilde = np.dot(A, D)  # Scale the Matrix A
    # print(A_tilde)
    b_tilde = b  # b does not change
    x0 = np.zeros_like(b)  # Initialize the Starting Guess

    # Solve the scaled system using Gauss-Seidel
    x_tilde, num_iterations = gauss_seidel(A_tilde, b_tilde, x0=x0, tol=tol, max_iterations=max_iterations)

    # Solve the scaled system using Jacobi
    # x_tilde, num_iterations = jacobi(A_tilde, b_tilde, x0=x0, tol=tol, max_iterations=max_iterations)

    # Solve the scaled system using GMRES(Generalized Minimal Residual) by defined LinearOperator
    # def mv(v):
    #     return np.dot(A_tilde, v)

    # A_op = LinearOperator(A_tilde.shape, matvec=mv)
    # x_tilde, num_iterations = gmres(A_op, b_tilde, x0=x0, tol=tol, maxiter=max_iterations)

    # Scale the solution back to the original system
    x = np.dot(D, x_tilde)
    # x = x_tilde

    return x, num_iterations

# Initial guess for the iterative methods
x0 = np.zeros_like(b)

# Solve using the Jacobi method
x_jacobi, it_jacobi = jacobi(A, b, x0)
err_jacobi = np.linalg.norm(x_jacobi - x_exact)

# Solve using the Gauss-Seidel method
x_gs, it_gs = gauss_seidel(A, b, x0)
err_gs = np.linalg.norm(x_gs - x_exact)

# Solve using the Multi-scale method
x_ms, it_ms = multi_scale(A, b)
err_ms = np.linalg.norm(x_ms - x_exact)

# Create a DataFrame to display the results
results = pd.DataFrame({
    'Method': ['Jacobi', 'Gauss-Seidel', 'Multi-scale'],
    'Iterations (IT)': [it_jacobi, it_gs, it_ms],
    'Relative Error (ERR)': [err_jacobi, err_gs, err_ms],
    'Final Solution': [x_jacobi, x_gs, x_ms]
})

print(results)

         Method  Iterations (IT)  Relative Error (ERR)  \
0        Jacobi                7          1.732141e-12   
1  Gauss-Seidel                5          9.992007e-15   
2   Multi-scale                5          9.769963e-15   

                                      Final Solution  
0  [0.9999999999989999, 0.999999999999, 0.9999999...  
1                       [0.99999999999999, 1.0, 1.0]  
2                     [0.9999999999999902, 1.0, 1.0]  


# **Computing the singular values of matrices in Example 1:**

In [ ]:
import numpy as np

# Define the original matrix A
A = np.array([
    [1, 1, 0],
    [0, 100, 1],
    [1, 0, 10000]
])

# Compute the singular values of the original matrix A
singular_values_A = np.linalg.svd(A, compute_uv=False)

# Compute the condition number of the original matrix A
cond_A = max(singular_values_A) / min(singular_values_A)

# Print the results
print(f"The original matrix A is:\n{A}")
print("\nThe singular values of A are: ", singular_values_A)
print(f"The condition number of A is: {cond_A}")

The original matrix A is:
[[    1     1     0]
 [    0   100     1]
 [    1     0 10000]]

The singular values of A are:  [1.00000001e+04 1.00005000e+02 9.99950994e-01]
The condition number of A is: 10000.490186030387


In [ ]:
import numpy as np
# Define the original matrix A
A = np.array([
[1, 1, 0],
[0, 100, 1],
[1, 0, 10000]
])
# Compute the diagonal scaling matrix D
D_values = np.linalg.norm(A, axis=0)
D = np.diag(1 / D_values)
# Compute the scaled matrix A_tilde
A_tilde = np.dot(A, D)
# Compute the singular values of the scaled matrix A_tilde
singular_values = np.linalg.svd(A_tilde, compute_uv=False)
# Compute the condition number
cond_A_tilde = max(singular_values) / min(singular_values)
print(f"The scaled matrix A_tilde is:\n{A_tilde}")
print("\nThe singular_values of A_tilde is: ", singular_values)
print(f"The condition number of A_tilde is: {cond_A_tilde}")

The scaled matrix A_tilde is:
[[7.07106781e-01 9.99950004e-03 0.00000000e+00]
 [0.00000000e+00 9.99950004e-01 9.99999995e-05]
 [7.07106781e-01 0.00000000e+00 9.99999995e-01]]

The singular_values of A_tilde is:  [1.30657688 0.999999   0.54116436]
The condition number of A_tilde is: 2.4143808690684616


# **Example 2. Finding exact solution:**

In [ ]:
import numpy as np

# Define the matrix A
A = np.array([[1, 1/2, 1/3],
              [1/2, 1/3, 1/4],
              [1/3, 1/4, 1/5]])

# Define the vector b
b = np.array([1, 1, 1])

# Solve the linear system Ax = b
x = np.linalg.solve(A, b)

# Print the solution
print("Real solution: ")
print(x)

Real solution: 
[  3. -24.  30.]


# **Solving Example 2 using Jacobi, Gauss-Siedel, and Multi-scale methods for comparison:**

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import LinearOperator, gmres

# Define the coefficient matrix A and the right-hand side vector b
A = np.array([[1, 1/2, 1/3],
              [1/2, 1/3, 1/4],
              [1/3, 1/4, 1/5]])

# Define the vector b
b = np.array([1, 1, 1])

# Define the exact solution for comparison
x_exact = np.linalg.solve(A, b)

# Define a function for the Jacobi method
def jacobi(A, b, x0, tol=1e-10, max_iterations=1000):
    n = len(b)
    x = x0.copy()
    D = np.diag(A)
    R = A - np.diagflat(D)
    for i in range(max_iterations):
        x_new = (b - np.dot(R, x)) / D
        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            break
        x = x_new
    return x, i+1

# Define a function for the Gauss-Seidel method
def gauss_seidel(A, b, x0=None, tol=1e-10, max_iterations=1000):
    n = len(b)
    if x0 is None:
        x0 = np.zeros(n)

    x = x0.copy()
    D = np.diag(np.diag(A))
    L = np.tril(A, -1)
    U = np.triu(A, 1)

    for k in range(max_iterations):
        x_new = np.dot(np.linalg.inv(D + L), b - np.dot(U, x))

        if np.linalg.norm(x_new - x, ord=np.inf) < tol:
            return x_new, k
        x = x_new

    return x, max_iterations


# Define a function for the Multi-scale method using the Gauss-Seidel, Jacobi, GMRES methods
def multi_scale(A, b, tol=1e-10, max_iterations=1000):
    D = np.linalg.norm(A, axis=0)  # Compute the Diagonal Scaling Matrix D
    D = np.diag(D)
    A_tilde = np.dot(A, D)  # Scale the Matrix A
    # print(A_tilde)
    b_tilde = b  # b does not change
    x0 = np.zeros_like(b)  # Initialize the Starting Guess

    # Solve the scaled system using Gauss-Seidel
    x_tilde, num_iterations = gauss_seidel(A_tilde, b_tilde, x0=x0, tol=tol, max_iterations=max_iterations)

    # Solve the scaled system using Jacobi
    # x_tilde, num_iterations = jacobi(A_tilde, b_tilde, x0=x0, tol=tol, max_iterations=max_iterations)

    # Solve the scaled system using GMRES(Generalized Minimal Residual) by defined LinearOperator
    # def mv(v):
    #     return np.dot(A_tilde, v)

    # A_op = LinearOperator(A_tilde.shape, matvec=mv)
    # x_tilde, num_iterations = gmres(A_op, b_tilde, x0=x0, tol=tol, maxiter=max_iterations)

    # Scale the solution back to the original system
    x = np.dot(D, x_tilde)
    # x = x_tilde

    return x, num_iterations

# Initial guess for the iterative methods
x0 = np.zeros_like(b)

# Solve using the Jacobi method
x_jacobi, it_jacobi = jacobi(A, b, x0)
err_jacobi = np.linalg.norm(x_jacobi - x_exact)

# Solve using the Gauss-Seidel method
x_gs, it_gs = gauss_seidel(A, b, x0)
err_gs = np.linalg.norm(x_gs - x_exact)

# Solve using the Multi-scale method
x_ms, it_ms = multi_scale(A, b)
err_ms = np.linalg.norm(x_ms - x_exact)

# Create a DataFrame to display the results
results = pd.DataFrame({
    'Method': ['Jacobi', 'Gauss-Seidel', 'Multi-scale'],
    'Iterations (IT)': [it_jacobi, it_gs, it_ms],
    'Relative Error (ERR)': [err_jacobi, err_gs, err_ms],
    'Final Solution': [x_jacobi, x_gs, x_ms]
})

print(results)

         Method  Iterations (IT)  Relative Error (ERR)  \
0        Jacobi             1000                   inf   
1  Gauss-Seidel             1000          1.650781e-07   
2   Multi-scale             1000          1.650779e-07   

                                      Final Solution  
0  [-1.095738359843396e+236, -2.0686415929498956e...  
1  [2.9999999764793808, -23.9999998799592, 29.999...  
2  [2.9999999764793963, -23.999999879959315, 29.9...  


# **Computing the singular values of matrices in Example 2:**

In [ ]:
import numpy as np

# Define the original matrix A
A = np.array([[1, 1/2, 1/3],
              [1/2, 1/3, 1/4],
              [1/3, 1/4, 1/5]])

# Compute the singular values of the original matrix A
singular_values_A = np.linalg.svd(A, compute_uv=False)

# Compute the condition number of the original matrix A
cond_A = max(singular_values_A) / min(singular_values_A)

# Print the results
print(f"The original matrix A is:\n{A}")
print("\nThe singular values of A are: ", singular_values_A)
print(f"The condition number of A is: {cond_A}")

The original matrix A is:
[[1.         0.5        0.33333333]
 [0.5        0.33333333 0.25      ]
 [0.33333333 0.25       0.2       ]]

The singular values of A are:  [1.40831893 0.12232707 0.00268734]
The condition number of A is: 524.0567775860644


In [ ]:
import numpy as np
# Define the original matrix A
A = np.array([[1, 1/2, 1/3],
              [1/2, 1/3, 1/4],
              [1/3, 1/4, 1/5]])

# Compute the diagonal scaling matrix D
D_values = np.linalg.norm(A, axis=0)
D = np.diag(1 / D_values)
# Compute the scaled matrix A_tilde
A_tilde = np.dot(A, D)
# Compute the singular values of the scaled matrix A_tilde
singular_values = np.linalg.svd(A_tilde, compute_uv=False)
# Compute the condition number
cond_A_tilde = max(singular_values) / min(singular_values)
print(f"The scaled matrix A_tilde is:\n{A_tilde}")
print("\nThe singular_values of A_tilde is: ", singular_values)
print(f"The condition number of A_tilde is: {cond_A_tilde}")

The scaled matrix A_tilde is:
[[0.85714286 0.76822128 0.72121845]
 [0.42857143 0.51214752 0.54091383]
 [0.28571429 0.38411064 0.43273107]]

The singular_values of A_tilde is:  [1.72408534 0.16585673 0.0046133 ]
The condition number of A_tilde is: 373.72034839797374


# To download

In [ ]:
# !sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic

In [ ]:
# !jupyter nbconvert --to pdf /content/Math548_hw6_Lazizbek.ipynb